# 3　主要过程

按论文顺序完成第 II-F 折内融合与关联强度分级、II-G 实验设计与评价，以及第 III 章的全部结果：
各方法性能对比与配对检验、特征消融、共线性、敏感性、同档内排序，直至全库评分与相对分层。

用到的输入都来自 1 号（`结果/全库候选对_结构特征.xlsx`、`结果/专家参考标注_243对.xlsx`、
`结果/网络边表.xlsx`）和 2 号（`结果/全库语义特征.xlsx`、`结果/243对语义特征.xlsx`）。

In [ ]:
import os
from pathlib import Path

# 找到工程根目录（含「数据」文件夹的那一层）
ROOT = Path(os.path.abspath(".")).resolve()
while ROOT != ROOT.parent and not (ROOT / "数据").is_dir():
    ROOT = ROOT.parent
DATA = ROOT / "数据"
RES = ROOT / "结果"
RES.mkdir(exist_ok=True)

import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, roc_auc_score
from sklearn.model_selection import KFold
from statsmodels.stats.contingency_tables import mcnemar

KEY_COLS = ["症状", "病机"]

FEATURE_COLS = ["shared_syndrome_count_norm", "NPMI_norm", "Jaccard_norm",
                "AA_norm", "RA_norm", "PPR_norm"]
SEM_COL = "llm_norm"                 # 第 7 项语义特征
FEATURE_COLS7 = FEATURE_COLS + [SEM_COL]
FEATURE_CN = {"shared_syndrome_count_norm": "共享证候数", "NPMI_norm": "NPMI",
              "Jaccard_norm": "Jaccard", "AA_norm": "AA", "RA_norm": "RA", "PPR_norm": "PPR"}
names = [FEATURE_CN[c] for c in FEATURE_COLS] + ["LLM语义"]

RIDGE_ALPHA = 1.0     # 岭回归正则化参数 λ
N_SPLITS = 5          # 折数
N_REPEATS = 300       # 重复次数，合计 1500 折
RANDOM_STATE = 42     # 第 r 次重复用 RANDOM_STATE + r


## 3.1　折内融合与分级（II-F）

训练折里用岭回归学 7 个特征的权重，负系数截为 0 后归一化；再按训练折标签的类别比例取 4 个
分级阈值，把连续分值映射成五级。测试折只做前向应用，不参与任何拟合。

In [ ]:
def fit_ridge_weights(X, y):
    """折内学权重：岭回归系数把负的截掉，剩下的归一化。"""
    model = Ridge(alpha=RIDGE_ALPHA).fit(X, y)
    w = np.clip(model.coef_, 0, None)
    if w.sum() == 0:
        w = np.abs(model.coef_)
    return w / w.sum()


def learn_thresholds_by_distribution(scores_train, y_train):
    """训练折内按标签类别比例取 4 个分级阈值。

    类别比例 p1..p5 累加，阈值取在 p1、p1+p2、p1+p2+p3、p1+p2+p3+p4 分位处。
    """
    props = np.array([np.mean(y_train == c) for c in [1, 2, 3, 4, 5]])
    cum_pcts = np.cumsum(props)[:-1] * 100
    return np.percentile(scores_train, cum_pcts)


def score_to_level(scores, thresholds):
    """连续分值按阈值映射为五级。"""
    return np.digitize(scores, thresholds) + 1


def run_cv(X, y, n_repeats=N_REPEATS):
    """5 折重复 n_repeats 次，每次换一个随机种子。

    训练折里学权重、取阈值，测试折只做前向应用。
    连续分值取多次重复的均值，五级预测取众数。
    """
    X = np.asarray(X, float); y = np.asarray(y, int)
    n = len(y)
    stack = np.zeros((n, n_repeats), int)
    score = np.zeros((n, n_repeats))
    weights = np.zeros((n_repeats * N_SPLITS, X.shape[1]))
    k = 0
    for r in range(n_repeats):
        for tr, te in KFold(N_SPLITS, shuffle=True, random_state=RANDOM_STATE + r).split(X):
            w = fit_ridge_weights(X[tr], y[tr])
            weights[k] = w; k += 1
            s_tr, s_te = X[tr] @ w, X[te] @ w
            stack[te, r] = score_to_level(s_te, learn_thresholds_by_distribution(s_tr, y[tr]))
            score[te, r] = s_te
    pred = np.array([np.bincount(stack[i], minlength=6)[1:].argmax() + 1 for i in range(n)])
    return pred, score.mean(axis=1), weights


def evaluate_full(y_true, y_pred, y_cont):
    """论文 II-G 的评价指标。"""
    y_true = np.asarray(y_true)
    y_pred = np.clip(np.rint(y_pred), 1, 5).astype(int)
    d = y_true - y_pred
    return {"Acc(精确)": round(accuracy_score(y_true, y_pred), 4),
            "Acc(±1容差)": round((np.abs(d) <= 1).mean(), 4),
            "F1-macro": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4),
            "quadratic-κ": round(cohen_kappa_score(y_true, y_pred, weights="quadratic"), 4),
            "Spearman": round(spearmanr(y_cont, y_true).statistic, 4),
            "KendallTau": round(kendalltau(y_true, y_cont).statistic, 4),
            "AUC(≥中等)": round(roc_auc_score((y_true >= 3).astype(int), y_cont), 4),
            "AUC(≥强)": round(roc_auc_score((y_true >= 4).astype(int), y_cont), 4),
            "分值取值数": int(len(np.unique(np.round(np.asarray(y_cont, float), 12))))}

## 3.2　拼 7 维特征

243 对参考样本按症状—病机键把六项结构特征和第 7 项语义特征拼在一起。

In [ ]:
df_labeled = pd.read_excel(RES / "专家参考标注_243对.xlsx")
df_scores = pd.read_excel(RES / "全库候选对_结构特征.xlsx")
df_sem = pd.read_excel(RES / "全库语义特征.xlsx")
for d in (df_labeled, df_scores, df_sem):
    for c in KEY_COLS:
        d[c] = d[c].astype(str).str.strip()

df_gold = (df_labeled[KEY_COLS + ["专家1", "专家2", "专家3", "y"]]
           .merge(df_scores[KEY_COLS + FEATURE_COLS], on=KEY_COLS, how="left")
           .merge(df_sem[KEY_COLS + ["llm_level", SEM_COL]], on=KEY_COLS, how="left"))
assert len(df_gold) == len(df_labeled) and df_gold[FEATURE_COLS7].notna().all().all(), "特征有缺失"

y = df_gold["y"].values.astype(int)
X = df_gold[FEATURE_COLS7].values.astype(float)
print("参考样本 %d 对；标签分布 %s" % (len(df_gold), df_gold["y"].value_counts().sort_index().to_dict()))

## 3.3　主方案的交叉验证结果

5 折 × 300 次重复，每次重复换一个随机种子，共 1500 折。同时给出折内权重和折内阈值。

In [ ]:
pred, cont, weights = run_cv(X, y)
met = evaluate_full(y, pred, cont)
for k, v in met.items():
    print("  %-12s %s" % (k, v))

d = y - pred
print("完全一致 %d，相差 1 级 %d，相差 2 级及以上 %d（高估 %d、低估 %d）"
      % ((d == 0).sum(), (np.abs(d) == 1).sum(), (np.abs(d) >= 2).sum(),
         (d <= -2).sum(), (d >= 2).sum()))
cm = np.zeros((5, 5), int)
for i in range(len(y)):
    cm[y[i] - 1, pred[i] - 1] += 1
print("混淆矩阵（行=专家等级，列=预测等级）")
for row in cm:
    print("   " + "  ".join("%3d" % v for v in row))

print("折内权重（1500 折均值 ± 标准差）")
for i, n in enumerate(names):
    print("  %-8s %.4f ± %.4f" % (n, weights[:, i].mean(), weights[:, i].std(ddof=1)))

In [ ]:
thr = []
for r in range(N_REPEATS):
    for tr, te in KFold(N_SPLITS, shuffle=True, random_state=RANDOM_STATE + r).split(X):
        thr.append(learn_thresholds_by_distribution(X[tr] @ fit_ridge_weights(X[tr], y[tr]), y[tr]))
TH = np.array(thr)
df_thr = pd.DataFrame([
    {"阈值": "t%d" % (j + 1),
     "含义": ["1/2 档分界", "2/3 档分界", "3/4 档分界", "4/5 档分界"][j],
     "均值": round(float(TH[:, j].mean()), 4),
     "标准差": round(float(TH[:, j].std()), 4),
     "跨折CV": round(float(TH[:, j].std() / abs(TH[:, j].mean())), 3)}
    for j in range(TH.shape[1])])
df_thr.to_excel(RES / "折内分级阈值.xlsx", index=False)
print(df_thr.to_string(index=False))

out = df_gold[KEY_COLS + ["专家1", "专家2", "专家3", "y"]].copy()
out["pred_五级"] = pred
out["score_融合"] = np.round(cont, 6)
for i, c in enumerate(FEATURE_COLS7):
    out[c] = X[:, i]
out.to_excel(RES / "243对_逐对预测.xlsx", index=False)

pd.DataFrame(weights, columns=names).to_excel(RES / "折内权重.xlsx", index=False)
pd.DataFrame([{"指标": k, "数值": v} for k, v in met.items()]
             + [{"指标": "权重·" + n, "数值": round(float(weights[:, i].mean()), 4)}
                for i, n in enumerate(names)]).to_excel(RES / "主结果指标.xlsx", index=False)
print("已写出 243对_逐对预测 / 折内权重 / 折内分级阈值 / 主结果指标")

## 3.4　表 III　各方法全口径性能

第一个对照是语言模型单独判定：直接把语义档位当作五级预测。第二个用 DeepSeek-V4-Flash
的判定做对照。第三个只用 6 项结构特征，走同一套折内融合与分级流程。

In [ ]:
df_T = out.copy()
y = df_T["y"].values.astype(int)
pred_main = df_T["pred_五级"].values.astype(int)
cont_main = df_T["score_融合"].values.astype(float)

sem243 = pd.read_excel(RES / "243对语义特征.xlsx")
for c in KEY_COLS:
    sem243[c] = sem243[c].astype(str).str.strip()
sem243 = df_T[KEY_COLS].merge(sem243[KEY_COLS + ["qwen_243", "deepseek_243"]],
                              on=KEY_COLS, how="left")
assert sem243[["qwen_243", "deepseek_243"]].notna().all().all(), "两个基线有缺失"
q243 = sem243["qwen_243"].values.astype(int)
ds243 = sem243["deepseek_243"].values.astype(int)

rows = []


def add(name, pred, cont=None):
    m = evaluate_full(y, pred, cont if cont is not None else pred.astype(float))
    rows.append(dict(方法=name, **m))
    return m


m_main = add("岭回归融合＋折内校准（主方案）", pred_main, cont_main)

q = df_T[SEM_COL].values * 4 + 1
q_int = np.round(q).astype(int)
m_llm = add("大语言模型单独判定", q_int, df_T[SEM_COL].values)
m_ds = add("对照语言模型（DeepSeek-V4-Flash）单独判定", ds243, ds243.astype(float))

pred6, cont6, _ = run_cv(df_T[FEATURE_COLS].values.astype(float), y)
m_6 = add("仅 6 项结构特征", pred6, cont6)

df_t3 = pd.DataFrame(rows)
df_t3.to_excel(RES / "表3_各方法性能.xlsx", index=False)
print(df_t3.to_string(index=False))

## 3.5　配对检验

配对 McNemar 精确检验比较五级预测是否判对，配对 Bootstrap 估准确率之差的 95% 区间。
主方案与语言模型单独判定的对比同时用两个判定批次做基线，看结论对批次有多敏感。

In [ ]:
def mcnemar_pair(y_true, pa, pb):
    """配对 McNemar 精确检验。返回（仅前者对、仅后者对、p）。"""
    ca, cb = (np.asarray(pa) == y_true), (np.asarray(pb) == y_true)
    n01 = int((~ca & cb).sum()); n10 = int((ca & ~cb).sum())
    tab = [[int((ca & cb).sum()), n01], [n10, int((~ca & ~cb).sum())]]
    return n10, n01, float(mcnemar(tab, exact=True).pvalue)


def bootstrap_diff(y_true, pa, pb, n_boot=2000, seed=7):
    """配对 Bootstrap，估准确率之差的 95% 区间。"""
    y_true = np.asarray(y_true); pa = np.asarray(pa); pb = np.asarray(pb)
    rng = np.random.default_rng(seed)
    n = len(y_true)
    vals = np.array([(pa[i] == y_true[i]).mean() - (pb[i] == y_true[i]).mean()
                     for i in (rng.integers(0, n, n) for _ in range(n_boot))], float)
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


n10, n01, p_mc = mcnemar_pair(y, pred_main, q_int)
lo, hi = bootstrap_diff(y, pred_main, q_int)
print("不一致：融合对、语言模型错 %d 对；语言模型对、融合错 %d 对" % (n10, n01))
print("McNemar 精确检验 P = %.4f" % p_mc)
print("准确率差 %.2f 个百分点，Bootstrap 95%% 区间 %.2f ~ %.2f"
      % (100 * (m_main["Acc(精确)"] - m_llm["Acc(精确)"]), 100 * lo, 100 * hi))

n10b, n01b, p_b = mcnemar_pair(y, pred_main, pred6)
lo6, hi6 = bootstrap_diff(y, pred_main, pred6)
n10d, n01d, p_d = mcnemar_pair(y, pred_main, ds243)

# 再拿 243 对上的独立判定批次做基线，看差值对判定批次有多敏感
m_q243 = evaluate_full(y, q243, q243.astype(float))
n10e, n01e, p_e = mcnemar_pair(y, pred_main, q243)

df_pair = pd.DataFrame([
    {"比较": "主方案 vs 语言模型单独判定（全库判定批次）",
     "差异(百分点)": round(100 * (m_main["Acc(精确)"] - m_llm["Acc(精确)"]), 2),
     "仅前者正确": n10, "仅后者正确": n01, "McNemar P": round(p_mc, 4),
     "Bootstrap 下": round(100 * lo, 2), "Bootstrap 上": round(100 * hi, 2)},
    {"比较": "主方案 vs 语言模型单独判定（243 对独立判定批次）",
     "差异(百分点)": round(100 * (m_main["Acc(精确)"] - m_q243["Acc(精确)"]), 2),
     "仅前者正确": n10e, "仅后者正确": n01e, "McNemar P": round(p_e, 4),
     "Bootstrap 下": None, "Bootstrap 上": None},
    {"比较": "主方案 vs 仅 6 项结构特征",
     "差异(百分点)": round(100 * (m_main["Acc(精确)"] - m_6["Acc(精确)"]), 2),
     "仅前者正确": n10b, "仅后者正确": n01b, "McNemar P": round(p_b, 4),
     "Bootstrap 下": round(100 * lo6, 2), "Bootstrap 上": round(100 * hi6, 2)},
    {"比较": "主方案 vs DeepSeek 单独判定",
     "差异(百分点)": round(100 * (m_main["Acc(精确)"] - m_ds["Acc(精确)"]), 2),
     "仅前者正确": n10d, "仅后者正确": n01d, "McNemar P": round(p_d, 4),
     "Bootstrap 下": None, "Bootstrap 上": None}])
print("243 对独立批次（千问）单独判定 Acc = %.4f，与主方案相差 %.2f 个百分点"
      % (m_q243["Acc(精确)"], 100 * (m_main["Acc(精确)"] - m_q243["Acc(精确)"])))
df_pair.to_excel(RES / "配对检验.xlsx", index=False)
print(df_pair.to_string(index=False))

## 3.6　表 IV　特征消融

每个配置都重新学权重、重新分级，再和全量 7 特征做配对 McNemar，最后统一 Holm 校正。

In [ ]:
def holm_adjust(pvals):
    """Holm 校正，返回校正后的 p 值，顺序与输入一致。"""
    m = len(pvals)
    order = sorted(range(m), key=lambda i: pvals[i])
    adj = [0.0] * m
    prev = 0.0
    for k, i in enumerate(order):
        prev = min(1.0, max(prev, (m - k) * pvals[i]))
        adj[i] = prev
    return adj


configs = {
    "全量 7 特征": FEATURE_COLS7,
    "去统计共现维度": ["Jaccard_norm", "AA_norm", "RA_norm", "PPR_norm", SEM_COL],
    "去集合相似性维度": ["shared_syndrome_count_norm", "NPMI_norm", "AA_norm", "RA_norm",
                        "PPR_norm", SEM_COL],
    "去网络结构维度": ["shared_syndrome_count_norm", "NPMI_norm", "Jaccard_norm",
                       "PPR_norm", SEM_COL],
    "去共享证候数": ["NPMI_norm", "Jaccard_norm", "AA_norm", "RA_norm", "PPR_norm", SEM_COL],
    "去 RA": ["shared_syndrome_count_norm", "NPMI_norm", "Jaccard_norm", "AA_norm",
              "PPR_norm", SEM_COL],
    "去 PPR": ["shared_syndrome_count_norm", "NPMI_norm", "Jaccard_norm", "AA_norm",
               "RA_norm", SEM_COL],
    "仅语义特征": [SEM_COL],
    "仅 6 项结构特征": FEATURE_COLS,
}

abl, preds_abl, pvals = [], {}, []
for name, cols in configs.items():
    p, s, _ = run_cv(df_T[cols].values.astype(float), y)
    preds_abl[name] = p
    d = y - p
    x10, x01, px = mcnemar_pair(y, pred_main, p)
    abl.append({"配置": name, "特征数": len(cols),
                "Acc(精确)": round(float((p == y).mean()), 4),
                "Acc(±1)": round(float((np.abs(d) <= 1).mean()), 4),
                "quadratic-κ": round(float(cohen_kappa_score(y, p, weights="quadratic")), 4),
                "全量胜/对方胜": "%d / %d" % (x10, x01), "原始 p": round(px, 4)})
    pvals.append(px)

for r, adj in zip(abl, holm_adjust(pvals)):
    r["Holm p"] = round(adj, 4)
    r["校正后显著"] = "是" if adj < 0.05 else "否"
for r in abl:
    if r["配置"] == "去 PPR":
        r["备注"] = ("与全量 7 特征的预测完全一致（0 次等级翻转）"
                   if (preds_abl["去 PPR"] == pred_main).all() else "与全量有差异")

df_t4 = pd.DataFrame(abl)
df_t4.to_excel(RES / "表4_消融实验.xlsx", index=False)
print(df_t4.to_string(index=False))

## 3.7　特征共线性 VIF

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_rows = []
for tag, cols in [("6 项结构特征", FEATURE_COLS), ("7 维特征（含语义）", FEATURE_COLS7)]:
    Xv = df_T[cols].values.astype(float)
    Xv = np.column_stack([np.ones(len(Xv)), Xv])
    for j, c in enumerate(cols, start=1):
        vif_rows.append({"模型": tag, "特征": FEATURE_CN.get(c, "LLM语义"),
                         "VIF": round(float(variance_inflation_factor(Xv, j)), 1)})
df_vif = pd.DataFrame(vif_rows)
df_vif.to_excel(RES / "特征共线性VIF.xlsx", index=False)
print(df_vif.to_string(index=False))

## 3.8　敏感性分析

正则化参数、归一化口径、等级编码三组对照。λ 和等级编码按论文口径用 5 折 × 10 次重复，
折内 Min–Max 那组用 5 折 × 300 次。这几组都不使用专家标签。

In [ ]:
def quick_cv(cols, n_rep=10, alpha=1.0, enc=None):
    """5 折重复 n_rep 次的快速口径；enc 给定时替换语义特征的数值编码。"""
    Xs = df_T[cols].values.astype(float).copy()
    if enc is not None:
        lv = np.round(df_T[SEM_COL].values * 4).astype(int) + 1
        Xs[:, -1] = np.array(enc)[lv - 1]
        lo_, hi_ = Xs[:, -1].min(), Xs[:, -1].max()
        Xs[:, -1] = (Xs[:, -1] - lo_) / (hi_ - lo_ + 1e-12)
    Xs, yy = np.asarray(Xs, float), np.asarray(y, int)
    stack = np.zeros((len(yy), n_rep), int)
    score = np.zeros((len(yy), n_rep))
    for r in range(n_rep):
        for tr, te in KFold(N_SPLITS, shuffle=True, random_state=RANDOM_STATE + r).split(Xs):
            model = Ridge(alpha=alpha).fit(Xs[tr], yy[tr])
            w = np.clip(model.coef_, 0, None)
            if w.sum() == 0:
                w = np.abs(model.coef_)
            w = w / w.sum()
            s_tr, s_te = Xs[tr] @ w, Xs[te] @ w
            stack[te, r] = score_to_level(s_te, learn_thresholds_by_distribution(s_tr, yy[tr]))
            score[te, r] = s_te
    p = np.array([np.bincount(stack[i], minlength=6)[1:].argmax() + 1 for i in range(len(yy))])
    return (round(float((p == yy).mean()), 4),
            round(float(cohen_kappa_score(yy, p, weights="quadratic")), 4),
            round(float(spearmanr(score.mean(1), yy).statistic), 4))


sens = []
for a in [0.01, 0.1, 1.0, 10.0, 100.0]:
    acc, k, rho = quick_cv(FEATURE_COLS7, alpha=a)
    sens.append({"类别": "岭回归 λ", "设定": "λ = %g" % a, "重复次数": 10,
                 "Acc": acc, "κ": k, "ρ": rho})

sens.append({"类别": "归一化", "设定": "全库 28544 对 Min–Max（主口径）", "重复次数": 300,
             "Acc": m_main["Acc(精确)"], "κ": m_main["quadratic-κ"], "ρ": m_main["Spearman"]})

raw6 = ["shared_syndrome_count", "NPMI", "Jaccard", "AA", "RA", "PPR"]
Xraw = np.column_stack([
    df_T[KEY_COLS].merge(df_scores[KEY_COLS + raw6], on=KEY_COLS, how="left")[raw6]
    .values.astype(float), df_T[SEM_COL].values.astype(float)])
st_fold = np.zeros((len(y), 300), int); sc_fold = np.zeros((len(y), 300))
for r in range(300):
    for tr, te in KFold(N_SPLITS, shuffle=True, random_state=RANDOM_STATE + r).split(Xraw):
        mn, mx = Xraw[tr].min(0), Xraw[tr].max(0)
        rg = np.where(mx - mn < 1e-12, 1, mx - mn)
        Xtr = (Xraw[tr] - mn) / rg
        Xte = np.clip((Xraw[te] - mn) / rg, 0, 1)
        w = fit_ridge_weights(Xtr, y[tr])
        s_tr, s_te = Xtr @ w, Xte @ w
        st_fold[te, r] = score_to_level(s_te, learn_thresholds_by_distribution(s_tr, y[tr]))
        sc_fold[te, r] = s_te
p_fold = np.array([np.bincount(st_fold[i], minlength=6)[1:].argmax() + 1 for i in range(len(y))])
sens.append({"类别": "归一化", "设定": "折内 Min–Max（极值只用训练折）", "重复次数": 300,
             "Acc": round(float((p_fold == y).mean()), 4),
             "κ": round(float(cohen_kappa_score(y, p_fold, weights="quadratic")), 4),
             "ρ": round(float(spearmanr(sc_fold.mean(1), y).statistic), 4)})

for tag, enc in [("等距 0/.25/.5/.75/1", [0, .25, .5, .75, 1]),
                 ("凸型 0/.1/.3/.6/1", [0, .1, .3, .6, 1]),
                 ("凹型 0/.4/.7/.9/1", [0, .4, .7, .9, 1]),
                 ("秩凹型 0/.5/.75/.9/1", [0, .5, .75, .9, 1])]:
    acc, k, rho = quick_cv(FEATURE_COLS7, enc=enc)
    sens.append({"类别": "等级编码", "设定": tag, "重复次数": 10, "Acc": acc, "κ": k, "ρ": rho})

df_sens = pd.DataFrame(sens)
df_sens.to_excel(RES / "敏感性分析.xlsx", index=False)
print(df_sens.to_string(index=False))

## 3.9　同档内排序与整体排序差异（III-D）

语义判定给出的是五档离散标签，同一档内部的样本无法区分。融合分值在语义档相同、
专家等级不同的样本对上还能不能判对方向，用配对 Bootstrap 和配对置换检验来看。

In [ ]:
def within_level_pairwise(grade, score, rel):
    """语义档相同、专家等级不同的样本对上，连续分值方向判对的比例（打平记 0.5）。"""
    num = den = 0.0
    for i in range(len(rel)):
        for j in range(i + 1, len(rel)):
            if grade[i] != grade[j] or rel[i] == rel[j]:
                continue
            den += 1
            hi = i if rel[i] > rel[j] else j
            lo = j if hi == i else i
            num += 1.0 if score[hi] > score[lo] else (0.5 if score[hi] == score[lo] else 0.0)
    return (num / den if den else np.nan), int(den)


n_pair_total = len(y) * (len(y) - 1) / 2
tie_llm = sum(v * (v - 1) / 2 for v in pd.Series(q_int).value_counts())
acc_w, n_w = within_level_pairwise(q_int, cont_main, y)
acc_w_llm, _ = within_level_pairwise(q_int, q_int.astype(float), y)

# 同级内部的成对排序准确率：2 000 次配对 Bootstrap + 5 000 次配对置换检验
rng = np.random.default_rng(0)
bs = []
for _ in range(2000):
    idx = rng.integers(0, len(y), len(y))
    v, den = within_level_pairwise(q_int[idx], cont_main[idx], y[idx])
    if den > 0:
        bs.append(v)
ci_w = np.percentile(bs, [2.5, 97.5])

rng = np.random.default_rng(0)
perm = []
for _ in range(5000):
    v, den = within_level_pairwise(q_int, cont_main, rng.permutation(y))
    if den > 0:
        perm.append(v)
p_perm = float((np.array(perm) >= acc_w).mean())

print("语义同级且专家等级不同的派生两两比较：%d 个（无序对）" % n_w)
print("同级内方向判对比例 %.4f（95%% CI %.4f ~ %.4f），置换检验 P = %.4f"
      % (acc_w, ci_w[0], ci_w[1], p_perm))
print("语义判定自身的同级比例：%.1f%%（同级时无法区分，等价于随机）" % (100 * tie_llm / n_pair_total))


def boot(seed, fn):
    r = np.random.default_rng(seed)
    out = []
    for _ in range(2000):
        i = r.choice(len(y), len(y), replace=True)
        try:
            out.append(fn(i))
        except Exception:
            out.append(np.nan)
    return np.array(out, float)


# 融合与语义判定的整体排序差异，两批判定各作一次基线
delta_rows = []
for tag, base in [("全库判定批次", q_int), ("243 对独立判定批次", q243)]:
    pt_rho = spearmanr(cont_main, y).statistic - spearmanr(base, y).statistic
    pt_auc = (roc_auc_score((y >= 3).astype(int), cont_main)
              - roc_auc_score((y >= 3).astype(int), base.astype(float)))
    d_rho = boot(3, lambda i, b=base: spearmanr(cont_main[i], y[i]).statistic
                 - spearmanr(b[i], y[i]).statistic)
    d_auc = boot(3, lambda i, b=base: roc_auc_score((y[i] >= 3).astype(int), cont_main[i])
                 - roc_auc_score((y[i] >= 3).astype(int), b[i].astype(float)))
    print("以%s为基线：ΔSpearman %+.4f（95%% CI %+.4f ~ %+.4f）；"
          "ΔAUC(≥中等) %+.4f（95%% CI %+.4f ~ %+.4f）"
          % (tag, pt_rho, np.nanpercentile(d_rho, 2.5), np.nanpercentile(d_rho, 97.5),
             pt_auc, np.nanpercentile(d_auc, 2.5), np.nanpercentile(d_auc, 97.5)))
    delta_rows.append({"语义基线": tag,
                       "ΔSpearman": "%.4f（%.4f~%.4f）" % (pt_rho, np.nanpercentile(d_rho, 2.5),
                                                        np.nanpercentile(d_rho, 97.5)),
                       "ΔAUC(≥中等)": "%.4f（%.4f~%.4f）" % (pt_auc, np.nanpercentile(d_auc, 2.5),
                                                          np.nanpercentile(d_auc, 97.5))})

df_h3 = pd.DataFrame([
    {"指标": "同级且专家异级的派生两两比较", "融合": str(n_w), "语言模型": str(n_w)},
    {"指标": "同级内方向判对比例", "融合": "%.4f（%.4f~%.4f）" % (acc_w, ci_w[0], ci_w[1]),
     "语言模型": "%.4f" % acc_w_llm},
    {"指标": "置换检验 P（5000 次）", "融合": "%.4f" % p_perm, "语言模型": "—"}])
df_h3.to_excel(RES / "连续分值与同档内排序.xlsx", index=False)
pd.DataFrame(delta_rows).to_excel(RES / "整体排序差异.xlsx", index=False)
print(df_h3.to_string(index=False))
print(pd.DataFrame(delta_rows).to_string(index=False))

## 3.10　全库评分与相对分层（III-D）

用 1500 折的平均权重给全库 28544 对候选算连续分值，再按候选自身分值的 99、95、80、50
分位切成五档，得到相对分层。

In [ ]:
df_w = pd.read_excel(RES / "折内权重.xlsx")
df_all = (df_scores[KEY_COLS + ["shared_syndrome_count", "NPMI", "Jaccard", "AA", "RA", "PPR"]
                    + FEATURE_COLS]
          .merge(df_sem[KEY_COLS + ["llm_level", SEM_COL]], on=KEY_COLS, how="left"))
assert df_all[FEATURE_COLS7].notna().all().all()

w_bar = df_w[names].values.mean(axis=0)
print("最终权重:", dict(zip(names, np.round(w_bar, 4))))
df_all["final_score"] = df_all[FEATURE_COLS7].values.astype(float) @ w_bar
print("全库候选对 %d 对" % len(df_all))

lab = np.array(["极弱关联", "弱关联", "中等关联", "强关联", "极强关联"])
tier_names = [("极强关联", "前 1%"), ("强关联", "1%~5%"), ("中等关联", "5%~20%"),
              ("弱关联", "20%~50%"), ("极弱关联", "后 50%")]

sc = df_all["final_score"].values.astype(float)
qs = np.percentile(sc, [50, 80, 95, 99])
df_all["关联强度等级"] = lab[np.digitize(sc, np.sort(qs))]
print("分层阈值:", ", ".join("%.4f" % v for v in np.sort(qs)))

df_tier = pd.DataFrame([
    {"等级": k, "分位区间": rng_txt, "对数": int((df_all["关联强度等级"] == k).sum()),
     "占比": "%.2f%%" % (100 * (df_all["关联强度等级"] == k).mean()),
     "分值下限": round(float(sc[df_all["关联强度等级"] == k].min()), 4),
     "分值上限": round(float(sc[df_all["关联强度等级"] == k].max()), 4)}
    for k, rng_txt in tier_names])
df_tier.to_excel(RES / "全库相对分层.xlsx", index=False)
print(df_tier.to_string(index=False))

## 3.11　全库排序表与典型例子

In [ ]:
out_all = df_all[KEY_COLS + ["shared_syndrome_count", "NPMI", "Jaccard", "AA", "RA", "PPR",
                            "llm_level", SEM_COL, "final_score", "关联强度等级"]].copy()
out_all = out_all.rename(columns={"llm_level": "LLM五级判定", SEM_COL: "LLM语义编码"})
out_all = out_all.sort_values("final_score", ascending=False).reset_index(drop=True)
out_all.insert(0, "排名", np.arange(1, len(out_all) + 1))
out_all.to_excel(RES / "全库28544对_分值.xlsx", index=False)

print("分值最高的 10 对：")
for _, r in out_all.head(10).iterrows():
    print("  %-10s — %-18s %.4f（共享证候 %d）"
          % (r["症状"], r["病机"], r["final_score"], r["shared_syndrome_count"]))

# 没进候选集的组合，看它两端各连了多少个证候、有没有共享
edge = pd.read_excel(RES / "网络边表.xlsx")
sym2syn, pat2syn = {}, {}
for _, r in edge.iterrows():
    if {r["源类型"], r["目标类型"]} == {"证候", "症状"}:
        z, v = (r["源名称"], r["目标名称"]) if r["源类型"] == "证候" else (r["目标名称"], r["源名称"])
        sym2syn.setdefault(v, set()).add(z)
    elif {r["源类型"], r["目标类型"]} == {"证候", "病机"}:
        z, v = (r["源名称"], r["目标名称"]) if r["源类型"] == "证候" else (r["目标名称"], r["源名称"])
        pat2syn.setdefault(v, set()).add(z)
for s_, p_ in [("失眠", "心肾不交")]:
    cs, cp = sym2syn.get(s_, set()), pat2syn.get(p_, set())
    print("%s 连接 %d 个证候，%s 连接 %d 个证候，共享 %d 个"
          % (s_, len(cs), p_, len(cp), len(cs & cp)))

cases = ["黄腻苔|湿热内蕴", "恶寒|风寒袭表", "脉滑数|湿热内蕴", "发热|风热袭表", "失眠|心肾不交"]
lines = ["| 症状 | 病机 | 共享证候数 | 融合分值 | 相对等级 | 是否在候选集 |",
         "|---|---|---|---|---|---|"]
for c in cases:
    s, p = c.split("|")
    row = df_all[(df_all["症状"] == s) & (df_all["病机"] == p)]
    if len(row):
        r = row.iloc[0]
        lines.append("| %s | %s | %d | %.4f | %s | 是 |"
                     % (s, p, r["shared_syndrome_count"], r["final_score"], r["关联强度等级"]))
    else:
        lines.append("| %s | %s | — | — | — | 否（没有共享证候，当前知识结构下不可评价） |" % (s, p))
with open(RES / "典型案例.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")
print("已写出 全库28544对_分值 / 全库相对分层 / 典型案例.md")